Data preparation for the lung dataset: downloaded from https://data.humancellatlas.org/hca-bio-networks/lung/atlases/lung-v1-0, An integrated cell atlas of the human lung in health and disease (core)

In [1]:
# Imports and load
import sys
from pathlib import Path
import os
import scanpy as sc
import numpy as np
from scipy import sparse

REPO_ROOT = Path("~/scRNA-cross-donor-generalization").expanduser()
os.chdir(REPO_ROOT)
sys.path.append(str(REPO_ROOT / "src"))

In [2]:
from scrna_benchmark.embedding import compute_pca, compute_harmony
from scrna_benchmark.filtering import summarize_celltype_support

adata = sc.read_h5ad("data/lung/lung.h5ad")
print(adata)
print("\nobs columns:", adata.obs.columns.tolist())
print("layers:", list(adata.layers.keys()))

AnnData object with n_obs × n_vars = 584944 × 27402
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'original_ann_level_1', 'original_ann_level_2', 'origi

In [3]:
print("\nobs columns:")
print(adata.obs.columns.tolist())
print("\nvar columns:")
print(adata.var.columns.tolist())
print("\nobsm keys:")
print(list(adata.obsm.keys()))
print("\nlayers:")
print(list(adata.layers.keys()))
print("\nraw exists:", adata.raw is not None)


obs columns:
['suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'original_ann_level_1', 'original_ann_level_2', 'original_ann_level_3', 'original_ann_level_4', 'ori

In [4]:
for col in [
    "donor_id",
    "cell_type",
    "ann_level_1",
    "ann_level_2",
    "ann_level_3",
    "ann_level_4",
    "ann_level_5",
    "ann_finest_level",
    "scanvi_label",
    "disease",
    "lung_condition",
    "is_primary_data",
    "study",
    "dataset",
    "sample",
    "assay",
    "sequencing_platform",
    "suspension_type",
    "tissue",
    "tissue_type",
    "tissue_level_2",
    "tissue_level_3",
]:
    if col in adata.obs.columns:
        print(f"\n=== {col} ===")
        print(adata.obs[col].value_counts(dropna=False).head(30))


=== donor_id ===
donor_id
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_483747NU_CZI02           36814
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_483747donor 2            28792
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_483747NU_CZI01           28029
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_483747donor 3            24667
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_483747GRO-09             16187
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_483747D353               15380
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_483747390C               14728
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_483747356C               13557
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_483747GRO-04             13476
homosapiens_None_2023_None_sikkemalisa_001_d10_1101_2022_03_10_483747D372               13264
homosapiens_None_2023_None_sikkem

In [5]:
def summarize_matrix_values(X, name, n=100000):
    vals = X.data[:n] if sparse.issparse(X) else X.ravel()[:n]
    print(f"\n{name}")
    print("min:", vals.min())
    print("max:", vals.max())
    print("mean:", vals.mean())
    print("integer-like:", np.allclose(vals, np.round(vals)))
    return vals

x_vals = summarize_matrix_values(adata.X, "adata.X")

if adata.raw is not None:
    raw_vals = summarize_matrix_values(adata.raw.X, "adata.raw.X")
else:
    raw_vals = None

for layer in adata.layers.keys():
    summarize_matrix_values(adata.layers[layer], f"layer: {layer}")

if adata.raw is not None:
    total_per_cell = np.array(adata.raw.X.sum(axis=1)).flatten()
    print("\n.raw.X total per cell:")
    print(f"min: {total_per_cell.min():.1f}")
    print(f"max: {total_per_cell.max():.1f}")
    print(f"mean: {total_per_cell.mean():.1f}")
    print("integer-like raw data:", np.allclose(adata.raw.X.data[:10000], np.round(adata.raw.X.data[:10000])) if sparse.issparse(adata.raw.X) else np.allclose(adata.raw.X.ravel()[:10000], np.round(adata.raw.X.ravel()[:10000])))


adata.X
min: 0.06866274
max: 9.979599
mean: 0.89874905
integer-like: False

adata.raw.X
min: 1.0
max: 5963.0
mean: 5.5068
integer-like: True



.raw.X total per cell:
min: 241.0
max: 167967.0
mean: 7790.8
integer-like raw data: True


In [6]:
# =========================
# Lung benchmark preparation
# =========================

import pandas as pd
import scipy.sparse as sp

adata_filt = adata.copy()

# disease == normal and lung_condition == Healthy for the main clean benchmark.
adata_filt = adata_filt[adata_filt.obs["disease"].astype(str).str.lower() == "normal"].copy()

if "lung_condition" in adata_filt.obs.columns:
    adata_filt = adata_filt[
        adata_filt.obs["lung_condition"].astype(str) == "Healthy"
    ].copy()

# Require donor ID
adata_filt = adata_filt[adata_filt.obs["donor_id"].notna()].copy()

print("After disease/condition filtering:")
print(adata_filt)
print("n donors:", adata_filt.obs["donor_id"].nunique())
print(adata_filt.obs["lung_condition"].value_counts(dropna=False))

After disease/condition filtering:
AnnData object with n_obs × n_vars = 523962 × 27402
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'original_ann_level

In [7]:
# Choose lung annotation level
label_col = "ann_level_4"

adata_filt.obs["cell_type"] = adata_filt.obs[label_col].astype(str)
adata_filt.obs["donor_id"] = adata_filt.obs["donor_id"].astype(str)

# sample_id
adata_filt.obs["sample_id"] = adata_filt.obs["sample"].astype(str)

# batch: use dataset, because it captures study + technology split better than study alone
adata_filt.obs["batch"] = adata_filt.obs["dataset"].astype(str)

# Remove unusable labels
bad_labels = ["None", "nan", "NaN", "unknown", "Rare"]
adata_filt = adata_filt[
    ~adata_filt.obs["cell_type"].isin(bad_labels)
].copy()

print("After metadata standardization:")
print(adata_filt)
print("\ncell_type counts:")
print(adata_filt.obs["cell_type"].value_counts())
print("\nbatch counts:")
print(adata_filt.obs["batch"].value_counts())
print("\nn donors:", adata_filt.obs["donor_id"].nunique())
print("n cell types:", adata_filt.obs["cell_type"].nunique())

After metadata standardization:
AnnData object with n_obs × n_vars = 441942 × 27402
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'original_ann_level_1'

In [8]:
support_df = summarize_celltype_support(
    adata_filt,
    celltype_col="cell_type",
    donor_col="donor_id",
    min_cells=500,
    min_donors=10,
)

print(support_df.to_string(index=False))

keep_cell_types = support_df.loc[support_df["keep"], "cell_type"].tolist()

adata_filt = adata_filt[
    adata_filt.obs["cell_type"].isin(keep_cell_types)
].copy()

print(f"\nAfter support filtering: {adata_filt.n_obs:,} cells")
print("n donors:", adata_filt.obs["donor_id"].nunique())
print("n cell types:", adata_filt.obs["cell_type"].nunique())
print(adata_filt.obs["cell_type"].value_counts())

                cell_type  n_cells  n_donors  keep_by_cell_count  keep_by_donor_coverage  keep
     Alveolar macrophages    64229        81                True                    True  True
               Suprabasal    40448        74                True                    True  True
            Basal resting    38717        74                True                    True  True
            Multiciliated    38675        99                True                    True  True
                   Goblet    38449        82                True                    True  True
                     Club    35972        79                True                    True  True
 Interstitial macrophages    32066        88                True                    True  True
              CD8 T cells    26699        83                True                    True  True
              CD4 T cells    18433        85                True                    True  True
      Classical monocytes    16537        84      

In [9]:
def subsample_by_group(adata, group_cols, max_per_group=200, seed=42):
    rng = np.random.default_rng(seed)
    cells_to_keep = []

    for _, idx in adata.obs.groupby(group_cols, observed=True).groups.items():
        idx = np.array(list(idx))
        if len(idx) > max_per_group:
            idx = rng.choice(idx, size=max_per_group, replace=False)
        cells_to_keep.extend(idx)

    return adata[cells_to_keep].copy()


adata_filt = subsample_by_group(
    adata_filt,
    group_cols=["donor_id", "cell_type"],
    max_per_group=100,
    seed=42,
)

print("After donor-cell-type subsampling:")
print(adata_filt)
print("n donors:", adata_filt.obs["donor_id"].nunique())
print("n cell types:", adata_filt.obs["cell_type"].nunique())
print(adata_filt.obs["cell_type"].value_counts())

After donor-cell-type subsampling:
AnnData object with n_obs × n_vars = 83966 × 27402
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'original_ann_level_

In [10]:
# Remove filtered genes if present
if "feature_is_filtered" in adata_filt.var.columns:
    adata_filt = adata_filt[
        :, ~adata_filt.var["feature_is_filtered"].astype(bool)
    ].copy()

# Filter genes based on current X
sc.pp.filter_genes(adata_filt, min_cells=20)

# Align raw counts to the current filtered gene set
counts_raw = adata_filt.raw[:, adata_filt.var_names].X.copy()

# Verify raw counts
vals = counts_raw.data[:100000] if sparse.issparse(counts_raw) else counts_raw.ravel()[:100000]
print("counts_raw min:", vals.min())
print("counts_raw max:", vals.max())
print("counts_raw mean:", vals.mean())
print("counts_raw integer-like:", np.allclose(vals, np.round(vals)))

# Use raw counts as starting point
adata_filt.layers["counts"] = counts_raw.copy()
adata_filt.X = counts_raw.copy()

# Normalize/log for HVG/PCA/Harmony
sc.pp.normalize_total(adata_filt, target_sum=1e4)
sc.pp.log1p(adata_filt)

print("After count recovery + normalization:")
print(adata_filt)

counts_raw min: 1.0
counts_raw max: 219.0
counts_raw mean: 2.03247
counts_raw integer-like: True
After count recovery + normalization:
AnnData object with n_obs × n_vars = 83966 × 22921
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lun

In [11]:
# Store full normalized/log expression before HVG subsetting
adata_filt.raw = adata_filt.copy()

sc.pp.highly_variable_genes(
    adata_filt,
    n_top_genes=1000,
    batch_key="batch",
)

print(f"HVGs selected: {adata_filt.var['highly_variable'].sum()}")

hvg_mask = adata_filt.var["highly_variable"].values

# Subset to HVGs
adata_filt = adata_filt[:, adata_filt.var["highly_variable"]].copy()

print(f"After HVG subset: {adata_filt.shape}")

HVGs selected: 1000
After HVG subset: (83966, 1000)


In [12]:
compute_pca(adata_filt, n_comps=50, random_state=42)

print(f"PCA computed: {adata_filt.obsm['X_pca'].shape}")

# Keep 15 PCs for consistency
adata_filt.obsm["X_pca"] = adata_filt.obsm["X_pca"][:, :15]

print(f"X_pca after truncation: {adata_filt.obsm['X_pca'].shape}")

PCA computed: (83966, 50)
X_pca after truncation: (83966, 15)


In [13]:
compute_harmony(
    adata_filt,
    batch_col="batch",
    basis="X_pca",
    key_added="X_harmony",
    n_comps=15,
)

print(f"X_harmony: {adata_filt.obsm['X_harmony'].shape}")

2026-06-01 17:47:37,219 - harmonypy - INFO - Running Harmony (PyTorch on cpu)
2026-06-01 17:47:37,220 - harmonypy - INFO -   Parameters:
2026-06-01 17:47:37,221 - harmonypy - INFO -     max_iter_harmony: 10
2026-06-01 17:47:37,222 - harmonypy - INFO -     max_iter_kmeans: 20
2026-06-01 17:47:37,223 - harmonypy - INFO -     epsilon_cluster: 1e-05
2026-06-01 17:47:37,224 - harmonypy - INFO -     epsilon_harmony: 0.0001
2026-06-01 17:47:37,225 - harmonypy - INFO -     nclust: 100
2026-06-01 17:47:37,226 - harmonypy - INFO -     block_size: 0.05
2026-06-01 17:47:37,227 - harmonypy - INFO -     lamb: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
2026-06-01 17:47:37,229 - harmonypy - INFO -     theta: [2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2. 2.]
2026-06-01 17:47:37,230 - harmonypy - INFO -     sigma: [0.1 0.1 0.1 0.1 0.1]...
2026-06-01 17:47:37,231 - harmonypy - INFO -     verbose: True
2026-06-01 17:47:37,232 - harmonypy - INFO -     random_state: 0
2026-06-01 17:47:37,233 - harmonypy - INFO -   Dat

X_harmony: (83966, 15)


In [14]:
import scvi

# Subset raw counts to HVG genes
adata_filt.layers["counts"] = counts_raw[:, hvg_mask]

scvi.settings.seed = 42

scvi.model.SCVI.setup_anndata(
    adata_filt,
    layer="counts",
    batch_key="batch",
)

model = scvi.model.SCVI(
    adata_filt,
    n_latent=15,
)

model.train(
    max_epochs=200,
    early_stopping=True,
)

adata_filt.obsm["X_scVI"] = model.get_latent_representation()

print(f"X_scVI: {adata_filt.obsm['X_scVI'].shape}")

[rank: 0] Seed set to 42
/users/xchen5/.local/lib/python3.9/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3 /users/xchen5/.local/lib/python3.9/site-packages/ip ...
GPU available: False, used: False
TPU available: False, using: 0 TPU cores
/users/xchen5/.local/lib/python3.9/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python3 /users/xchen5/.local/lib/python3.9/site-packages/ip ...


Epoch 200/200: 100%|███████████████████████████████████████| 200/200 [1:14:22<00:00, 21.81s/it, v_num=1, train_loss_step=247, train_loss_epoch=261]

`Trainer.fit` stopped: `max_epochs=200` reached.


Epoch 200/200: 100%|███████████████████████████████████████| 200/200 [1:14:22<00:00, 22.31s/it, v_num=1, train_loss_step=247, train_loss_epoch=261]
X_scVI: (83966, 15)


In [15]:
print("=== Lung Benchmark-Ready Object ===")
print(adata_filt)

for key in ["X_pca", "X_harmony", "X_scVI"]:
    if key in adata_filt.obsm:
        print(f"{key}: {adata_filt.obsm[key].shape}")

print(f"\ncell_type: {adata_filt.obs['cell_type'].nunique()} types")
print(f"donor_id:  {adata_filt.obs['donor_id'].nunique()} donors")
print(f"batch:     {adata_filt.obs['batch'].nunique()} batches")
print(f"cells:     {adata_filt.n_obs:,}")

print("\nRequired obs columns:")
print(adata_filt.obs[["donor_id", "cell_type", "batch", "sample_id"]].head())

print("\ncell_type counts:")
print(adata_filt.obs["cell_type"].value_counts())

print("\ndonor counts summary:")
print(adata_filt.obs["donor_id"].value_counts().describe())

print("\nbatch counts:")
print(adata_filt.obs["batch"].value_counts())

=== Lung Benchmark-Ready Object ===
AnnData object with n_obs × n_vars = 83966 × 1000
    obs: 'suspension_type', 'donor_id', 'is_primary_data', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'tissue_ontology_term_id', 'sex_ontology_term_id', 'BMI', 'age_or_mean_of_age_range', 'age_range', 'anatomical_region_ccf_score', 'ann_coarse_for_GWAS_and_modeling', 'ann_finest_level', 'ann_level_1', 'ann_level_2', 'ann_level_3', 'ann_level_4', 'ann_level_5', 'cause_of_death', 'dataset', 'entropy_dataset_leiden_3', 'entropy_original_ann_level_1_leiden_3', 'entropy_original_ann_level_2_clean_leiden_3', 'entropy_original_ann_level_3_clean_leiden_3', 'entropy_subject_ID_leiden_3', 'fresh_or_frozen', 'leiden_1', 'leiden_2', 'leiden_3', 'leiden_4', 'leiden_5', 'log10_total_counts', 'lung_condition', 'mixed_ancestry', 'n_genes_detected', 'original_ann_highest_res', 'original_ann_level_

In [16]:
out_path = Path("data/lung/lung_benchmark_ready.h5ad")
out_path.parent.mkdir(parents=True, exist_ok=True)
adata_filt.write(out_path)

print(f"Saved: {out_path} ({out_path.stat().st_size / 1e6:.1f} MB)")

Saved: data/lung/lung_benchmark_ready.h5ad (1393.3 MB)
